In [0]:
%run "/Workspace/utilities"

In [0]:
dbutils.widgets.text("load_date", "")
dbutils.widgets.dropdown("load_type", "FULL", ["INCREMENTAL", "FULL"])

load_date = dbutils.widgets.get("load_date")
load_type = dbutils.widgets.get("load_type")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Configuration

# COMMAND ----------

# Use config from utilities
logger = DataLogger("transform_products")

source_path = config.get_source_path("products.csv")
silver_path = config.get_silver_path("products")

logger.info(f"Source path: {source_path}")
logger.info(f"Silver path: {silver_path}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Read Source Data

# COMMAND ----------

try:
    df_source = spark.read.format("csv") \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .load(source_path)
    
    logger.info(f"Read {df_source.count()} products from source CSV")
    display(df_source.limit(5))
    
except Exception as e:
    logger.error(f"Failed to read source data", e)
    raise

# COMMAND ----------

# MAGIC %md
# MAGIC ## Data Transformation

# COMMAND ----------

from pyspark.sql.functions import *

# Step 1: Clean and standardize
df_cleaned = df_source \
    .withColumn("product_name", trim(col("product_name"))) \
    .withColumn("category", upper(trim(col("category")))) \
    .withColumn("price", col("price").cast("decimal(10,2)"))

# Step 2: Validate price > 0
df_valid = df_cleaned.filter(col("price") > 0)

# Step 3: Remove duplicates
df_deduplicated = df_valid.dropDuplicates(["product_id"])

# Step 4: Add audit columns
df_final = df_deduplicated \
    .withColumn("created_date", current_timestamp()) \
    .withColumn("modified_date", current_timestamp())

logger.info(f"Transformed {df_final.count()} valid products")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Write to Silver Layer

# COMMAND ----------

try:
    df_final.write \
        .format("delta") \
        .mode("overwrite") \
        .save(silver_path)
    
    logger.info(f"✅ Successfully wrote {df_final.count()} products to Silver layer")
    
except Exception as e:
    logger.error(f"Failed to write to Silver layer", e)
    raise

# COMMAND ----------

# MAGIC %md
# MAGIC ## Data Quality Summary

# COMMAND ----------

print(f"\n📊 Transformation Summary:")
print(f"Source records: {df_source.count()}")
print(f"Invalid price records removed: {df_cleaned.count() - df_valid.count()}")
print(f"Duplicates removed: {df_valid.count() - df_deduplicated.count()}")
print(f"Final records: {df_final.count()}")

display(df_final.groupBy("category").count().orderBy("count", ascending=False))
